# Non-simulation pipeline V&V: by-scenario results checks

Summary checks over the non-simulation half of the snakemake/papermill pipeline -- the 0400
anemia and 0500 NTD multiplication models and the 5000 rescaling/aggregation -- as materialized
in the committed by-scenario results CSVs. Version-free filename by convention: rerun against
each model iteration and commit.

1. structural sanity: every file loads, no NaNs, counts nonnegative, prevalences in [0, 1],
   quintiles complete, scenario sets match the configured comparisons,
2. averted burden by configured comparison: interventions should not increase burden,
3. cross-version regression: what moved since the previous committed results, file by file.

Parameterized: `results_ref` is the git ref to read the CSVs from (None = working tree, the
default for reruns after a pipeline run); `previous_ref` is the comparison baseline (None
skips the regression section).

In [1]:
results_ref = None  # git ref for the results CSVs; None reads the working tree
previous_ref = None  # git ref to diff against; None skips the cross-version section

In [2]:
# Parameters
results_ref = None
previous_ref = "0306a01"


In [3]:
import io
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path.cwd().parents[2]
RESULTS = "5000_analyze_results/results"


def list_csvs(ref):
    if ref is None:
        return sorted(str(p.relative_to(REPO)) for p in (REPO / RESULTS).glob("*/*/*.csv"))
    out = subprocess.run(["git", "ls-tree", "-r", "--name-only", ref, "--", RESULTS],
                         capture_output=True, text=True, cwd=REPO, check=True).stdout.split()
    return sorted(p for p in out if p.endswith(".csv"))


def read_csv(ref, path):
    if ref is None:
        return pd.read_csv(REPO / path)
    out = subprocess.run(["git", "show", f"{ref}:{path}"],
                         capture_output=True, text=True, cwd=REPO, check=True)
    return pd.read_csv(io.StringIO(out.stdout))


frames = {}
for p in list_csvs(results_ref):
    loc, veh, name = p.split("/")[-3:]
    frames[(loc, veh, name.replace("_by_scenario.csv", ""))] = read_csv(results_ref, p)
comparisons = pd.read_csv(REPO / "0050_config/location_vehicle_scenario_comparisons.csv")
print(f"{len(frames)} result files loaded from {results_ref or 'the working tree'}")
sorted(set(m for _, _, m in frames))

24 result files loaded from the working tree


['dalys',
 'maternal_disorders_incident_cases',
 'neonatal_deaths',
 'ntd_cases',
 'pregnant_anemia_prevalence',
 'prevalent_anemia_cases']

## 1. Structural sanity

Prevalence measures must lie in [0, 1]; everything else is a count (or DALYs) and must be
nonnegative and finite. Every (location, vehicle) needs all five wealth quintiles per scenario,
and every scenario named in the configured comparisons must be present.

In [4]:
issues = []
for (loc, veh, meas), df in frames.items():
    label = f"{loc}/{veh}/{meas}"
    if df.value.isna().any() or not np.isfinite(df.value).all():
        issues.append(f"{label}: NaN or non-finite values")
    if "prevalence" in meas:
        if ((df.value < 0) | (df.value > 1)).any():
            issues.append(f"{label}: prevalence outside [0, 1]")
    elif (df.value < 0).any():
        issues.append(f"{label}: negative values")
    group_cols = [c for c in df.columns if c not in ("wealth_quintile", "value")]
    quintile_counts = df.groupby(group_cols).wealth_quintile.agg(["nunique", "count"])
    if (quintile_counts["nunique"] != 5).any():
        issues.append(f"{label}: incomplete wealth quintiles")
    scenarios = set(df.scenario)
    needed = comparisons[(comparisons.location == loc) & (comparisons.vehicle == veh)]
    for s in set(needed.baseline) | set(needed.intervention):
        if s not in scenarios:
            issues.append(f"{label}: configured scenario '{s}' missing")
print("\n".join(issues) if issues else "all structural checks pass")
assert not issues

all structural checks pass


## 2. Averted burden by configured comparison

For each comparison in `0050_config/location_vehicle_scenario_comparisons.csv` and each measure,
averted = comparator total - intervention total (summed over quintiles; DALYs kept per entity).
Interventions should not increase burden: negative averted beyond -0.5% of the comparator is
flagged. Small negatives within that tolerance are realization noise on near-zero effects.

In [5]:
rows = []
for _, cmp_row in comparisons.iterrows():
    for (loc, veh, meas), df in frames.items():
        if (loc, veh) != (cmp_row.location, cmp_row.vehicle):
            continue
        entity_cols = [c for c in df.columns if c not in ("scenario", "wealth_quintile", "value")]
        totals = df.groupby(["scenario"] + entity_cols).value.sum()
        for key in totals.loc[cmp_row.baseline].index if entity_cols else [None]:
            base = totals[(cmp_row.baseline, key) if key else cmp_row.baseline]
            intv = totals[(cmp_row.intervention, key) if key else cmp_row.intervention]
            rows.append((loc, veh, meas, key or "", f"{cmp_row.baseline} -> {cmp_row.intervention}",
                         base, base - intv, (base - intv) / base if base else np.nan))
averted = pd.DataFrame(rows, columns=["location", "vehicle", "measure", "entity", "comparison",
                                      "comparator_total", "averted", "averted_frac"])
with pd.option_context("display.max_rows", 200, "display.width", 200):
    print(averted.round(4).to_string(index=False))
flagged = averted[averted.averted_frac < -0.005]
print("\nflagged (burden increases beyond tolerance):", "none" if flagged.empty else "")
if not flagged.empty:
    print(flagged.to_string(index=False))

location  vehicle                           measure             entity                       comparison  comparator_total      averted  averted_frac
   india     rice                             dalys             anemia                 zero -> baseline      2.053730e+07 1.064653e+06        0.0518
   india     rice                             dalys              lbwsg                 zero -> baseline      7.029470e+07 1.311446e+05        0.0019
   india     rice                             dalys maternal_disorders                 zero -> baseline      1.706236e+06 4.194510e+04        0.0246
   india     rice                             dalys                ntd                 zero -> baseline      1.473624e+06 4.823188e+04        0.0327
   india     rice maternal_disorders_incident_cases                                    zero -> baseline      1.036357e+07 2.370575e+05        0.0229
   india     rice                   neonatal_deaths                                    zero -> baseline   

## 3. Cross-version regression

Per-file movement vs `previous_ref`: the median and max relative difference across matching
rows, plus how many values switched between zero and nonzero. Interpret with the known
between-version movers in mind (see notes below); NTD files are deterministic given fixed
inputs and should sit at 0.0% unless an input or the 0500 model changed.

In [6]:
if previous_ref is None:
    print("previous_ref not set; skipping")
else:
    reg = []
    for (loc, veh, meas), new in frames.items():
        path = f"{RESULTS}/{loc}/{veh}/{meas}_by_scenario.csv"
        try:
            old = read_csv(previous_ref, path)
        except subprocess.CalledProcessError:
            reg.append((f"{loc}/{veh}/{meas}", "new file", "", "", ""))
            continue
        key = [c for c in new.columns if c != "value"]
        m = old.merge(new, on=key, suffixes=("_old", "_new"))
        if len(m) != len(new):
            reg.append((f"{loc}/{veh}/{meas}", f"structure changed ({len(m)}/{len(new)} rows match)", "", "", ""))
            continue
        denom = np.maximum(np.abs(m.value_old), np.abs(m.value_new))
        ok = denom > 1e-12
        rel = np.where(ok, np.abs(m.value_new - m.value_old) / denom.where(ok, 1), 0.0)
        flips = int(((np.abs(m.value_old) < 1e-12) != (np.abs(m.value_new) < 1e-12)).sum())
        reg.append((f"{loc}/{veh}/{meas}", "ok", f"{np.median(rel[ok]):.1%}" if ok.any() else "0%",
                    f"{rel.max():.1%}", flips or ""))
    print(pd.DataFrame(reg, columns=["file", "status", "median_rel_diff", "max_rel_diff", "zero_flips"])
          .to_string(index=False))

                                              file status median_rel_diff max_rel_diff zero_flips
                               ethiopia/salt/dalys     ok            0.0%        64.2%           
   ethiopia/salt/maternal_disorders_incident_cases     ok              0%         0.0%           
                     ethiopia/salt/neonatal_deaths     ok              0%         0.0%           
                           ethiopia/salt/ntd_cases     ok           23.8%        64.2%           
          ethiopia/salt/pregnant_anemia_prevalence     ok              0%         0.0%           
              ethiopia/salt/prevalent_anemia_cases     ok            0.0%         0.0%           
                                  india/rice/dalys     ok            0.0%        27.1%           
      india/rice/maternal_disorders_incident_cases     ok            0.0%         0.0%           
                        india/rice/neonatal_deaths     ok            0.0%         0.0%           
                    

## Notes: known movers and pending fixes affecting these outputs

- GBD best-version refreshes move the runtime population pulls (~+5% between the model-1.1 and
  model-1.1.1 runs), and the regenerated pregnancy-status population split moved the pregnant
  population much more (~+18%) -- both scale case counts independent of any code change, so
  judge levels against external targets, not the previous CSVs.
- Ethiopia's pregnancy-side inputs come through the rescaled-results template path; its
  numbers also depend on which maternal run (and how many seeds) the run marker picked up.
- Pending fixes that change these outputs when they land: the folate coverage fix (PR 20:
  Ethiopia anemia impact roughly doubles), the U5 consumption inflation fix (Nigeria child
  doses ~1.5-1.6x high until the consumption CSVs are regenerated), and the India rice
  2021-baseline deletion (zero-scenario hemoglobin overstated).
- The interpolator-scramble fix will move the neonatal-deaths and DALY files through the
  rebuilt child sims.